# Iftixor — Google Colab orqali GPU bilan o'qitish

**Ishlatish tartibi:**
1. `Runtime -> Change runtime type -> T4 GPU` ni tanlang
2. Chap paneldagi 🔑 (Secrets) belgisidan `OPENAI_API_KEY` nomli maxfiy kalit qo'shing, qiymatiga OpenAI kalitingizni yozing, "Notebook access"ni yoqing
3. Har bir katakchani **tepadan pastga qarab, tartib bilan** ishga tushiring (▶ yoki Shift+Enter)

Barcha katakchalar qayta ishga tushirilsa ham xato bermaydigan qilib yozilgan — sessiya uzilib qolib, qaytadan boshlashga to'g'ri kelsa ham, faqat tepadan pastga qarab qayta bosib chiqishingiz kifoya.

In [ ]:
import torch
print("GPU mavjud:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU nomi:", torch.cuda.get_device_name(0))
else:
    print("GPU topilmadi — Runtime -> Change runtime type -> T4 GPU ni tanlab, qayta ishga tushiring.")

## 1. Repozitoriyani yuklab olish

In [ ]:
import os

if not os.path.exists('/content/Iftixor'):
    !git clone https://github.com/Iftix0r/Iftixor.git /content/Iftixor
else:
    print("Iftixor papkasi allaqachon mavjud, eng so'nggi kodni tortib olamiz...")
    !git -C /content/Iftixor pull

%cd /content/Iftixor
!pip install -q python-dotenv openai

## 2. Google Drive'ga ulanish

Checkpoint shu yerga saqlanadi — Colab sessiyasi kutilmaganda uzilib qolsa ham (masalan uxlab qolsangiz), hech narsa yo'qolmaydi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/iftixor_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpointlar shu yerga saqlanadi:", CHECKPOINT_DIR)

## 3. (Ixtiyoriy) Serveringizdan qo'shimcha ma'lumot yuklash

Agar `data/conversations.txt` yoki boshqa matn faylini qo'shmoqchi bo'lsangiz, uni avval o'z kompyuteringizga yuklab oling, so'ng shu katakchani ishga tushirib faylni tanlang. Kerak bo'lmasa, bu katakchani **o'tkazib yuboring** (ishga tushirmang).

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()
for fname in uploaded.keys():
    shutil.move(fname, f"data/{fname}")
    print(f"data/{fname} ga joylandi")

## 4. OpenAI orqali sun'iy (synthetic) ma'lumot yaratish

3000 ta yangi namunaviy suhbat yaratadi (~15-20 daqiqa). Bu — sifatni oshirishning eng katta omili.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    raise SystemExit(
        "OPENAI_API_KEY topilmadi. Chap paneldagi \U0001F511 belgisidan "
        "'OPENAI_API_KEY' nomli maxfiy kalit qo'shing va 'Notebook access'ni yoqing."
    )

!python -m iftixor.gen_synthetic_data --count 3000 --batch-size 25

## 5. O'qitish — kattaroq model, GPU bilan

Taxminan 40-60 daqiqa. Natija to'g'ridan-to'g'ri Google Drive'ga saqlanadi.

In [ ]:
!python -m iftixor.train \
  --data data/corpus.txt data/synthetic.txt \
  --steps 15000 \
  --batch-size 128 \
  --n-layer 8 \
  --n-head 8 \
  --n-embd 384 \
  --block-size 256 \
  --device auto \
  --out /content/drive/MyDrive/iftixor_checkpoints/iftixor.pt

## 6. Terminalda tez sinash

`_best` — val loss eng past bo'lgan (eng kam "yodlab olgan") versiya, shuni sinaymiz.

In [ ]:
from iftixor.generate import load_model, generate_text

model, tokenizer = load_model('/content/drive/MyDrive/iftixor_checkpoints/iftixor_best.pt')

for savol in ["Salom!", "Ismingiz nima?", "O'zbekiston poytaxti qayer?", "Hazil qiling!"]:
    prompt = f"Foydalanuvchi: {savol}\nIftixor:"
    out = generate_text(model, tokenizer, prompt, max_new_tokens=100, temperature=0.7, top_k=40)
    reply = out[len(prompt):].split("Foydalanuvchi:")[0].strip()
    print(f">> {savol}\n   {reply}\n")

## 7. Tayyor checkpointni yuklab olish

Fayl allaqachon Google Drive'da xavfsiz saqlangan (`MyDrive/iftixor_checkpoints/iftixor_best.pt`). Xohlasangiz, to'g'ridan-to'g'ri kompyuteringizga ham yuklab olishingiz mumkin:

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/iftixor_checkpoints/iftixor_best.pt')

Kompyuteringizga tushgan faylni serveringizga joylashtiring:

```bash
scp iftixor_best.pt root@<SERVER_IP>:~/Iftixor/Github/Iftixor/checkpoints/iftixor.pt
```

So'ng Telegram'da botga `/reload` deb yozing.